# Дообучение multilingual-e5-base на парах «запрос -> объявление» (Kaggle)
Пары только из истории P1 с прореживанием (`src/make_pairs.py`), поэтому P2/P3 остаются честными.
Loss: CachedMultipleNegativesRankingLoss (остальные объявления батча служат негативами, cached-вариант даёт большой батч при малой памяти).
Сэмплер NO_DUPLICATES: без него одно объявление может оказаться негативом самому себе.
**Первый запуск с `SPEED_TEST = True`** (100 шагов, замер скорости), затем `False` для полной эпохи.

In [ ]:
# Те же версии, что локально (requirements-lock.txt): иначе веса и токенизация могут разойтись
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 datasets==5.0.1 accelerate==1.15.0

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # одна GPU: CachedMNRL с DataParallel ведёт себя плохо
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SPEED_TEST = True                  # True: 100 шагов и прогноз времени эпохи; False: полная эпоха
LOCAL_SMOKE = os.environ.get("LOCAL_SMOKE") == "1"   # проверка ноутбука на маке, на Kaggle не трогать

MODEL = "intfloat/multilingual-e5-small" if LOCAL_SMOKE else "intfloat/multilingual-e5-base"
MAX_LEN = 128                      # как у dense-канала на инференсе
BATCH = 16 if LOCAL_SMOKE else 512 # чем больше батч, тем больше in-batch негативов
MINI_BATCH = 8 if LOCAL_SMOKE else 64
LR, WARMUP, SEED = 2e-5, 0.1, 42
MAX_STEPS = 2 if LOCAL_SMOKE else (100 if SPEED_TEST else -1)

import glob
INPUT = (glob.glob("artifacts/kaggle/pairs_p1.parquet") if LOCAL_SMOKE
         else glob.glob("/kaggle/input/**/pairs_p1.parquet", recursive=True))[0]
WORK = "/tmp/e5_smoke" if LOCAL_SMOKE else "/kaggle/working"
print(INPUT, WORK)

In [ ]:
import json, time
import torch, transformers, datasets, sentence_transformers
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments, BatchSamplers
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss

assert sentence_transformers.__version__ == "6.1.0", sentence_transformers.__version__
print("st", sentence_transformers.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
transformers.set_seed(SEED)

In [ ]:
# Только колонки anchor и positive: trainer подаёт в loss все колонки, кроме label
train = datasets.Dataset.from_parquet(INPUT)
if LOCAL_SMOKE:
    train = train.select(range(200))
print(train)
print(train[0])

In [ ]:
model = SentenceTransformer(MODEL)
model.max_seq_length = MAX_LEN
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH)

args = SentenceTransformerTrainingArguments(
    output_dir=f"{WORK}/ckpt",
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH,
    learning_rate=LR,
    warmup_steps=WARMUP,                # в transformers 5 доля прогрева задаётся float в warmup_steps
    use_cpu=LOCAL_SMOKE,                # на MPS sdpa не поддерживает dropout; на Kaggle не влияет
    fp16=torch.cuda.is_available(),     # T4/P100 не умеют bf16
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    dataloader_drop_last=True,
    seed=SEED,
    data_seed=SEED,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,                 # чекпоинты в /kaggle/working на случай обрыва
    report_to="none",
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=loss)

In [ ]:
t0 = time.time()
trainer.train()
dt = time.time() - t0
steps = trainer.state.global_step
steps_epoch = len(train) // BATCH
print(f"{steps} шагов за {dt / 60:.1f} мин, {dt / steps:.2f} с/шаг")
print(f"прогноз полной эпохи ({steps_epoch} шагов): {dt / steps * steps_epoch / 3600:.2f} ч")
json.dump({"log_history": trainer.state.log_history, "sec_per_step": dt / steps, "steps": steps,
           "model": MODEL, "batch": BATCH, "mini_batch": MINI_BATCH, "lr": LR, "seed": SEED},
          open(f"{WORK}/train_log.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Итоговые веса: скачать из Output и положить в artifacts/e5-base-ft
if not SPEED_TEST or LOCAL_SMOKE:
    model.save(f"{WORK}/e5-base-ft")
    print(os.listdir(f"{WORK}/e5-base-ft"))